# Expresso SPARC Encoding
Encode Meta's Expresso dataset (40h, 7 styles, 4 speakers) into SPARC articulatory features.

Output: `expresso_features/*.npz` on Google Drive — each file has `ema(T,12)`, `pitch(T,)`, `loudness(T,)`, `spk_emb(64,)`, `style`, `text`, `speaker_id`, `utt_id`.

In [ ]:
# Cell 1: Mount Drive + check resume point
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
OUT_DIR = '/content/drive/MyDrive/expresso_features'
os.makedirs(OUT_DIR, exist_ok=True)
existing = set(f.replace('.npz','') for f in os.listdir(OUT_DIR) if f.endswith('.npz'))
print(f'Existing files on Drive: {len(existing)}')

In [ ]:
# Cell 2: Install SPARC + datasets
!pip install -q speech-articulatory-coding datasets soundfile

In [ ]:
# Cell 3: Load SPARC model on GPU
from sparc import load_model as load_sparc
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sparc = load_sparc('en', device=device)
print(f'SPARC loaded on {device}, sr={sparc.sr}')

In [ ]:
# Cell 4: Load Expresso dataset (streaming to avoid full download)
from datasets import load_dataset

ds = load_dataset('ylacombe/expresso', split='train')
print(f'Total samples: {len(ds)}')
print(f'Styles: {sorted(set(ds["style"]))}')
print(f'Speakers: {sorted(set(ds["speaker_id"]))}')

In [ ]:
# Cell 5: Encode all samples
import numpy as np
import time
import librosa

t0 = time.time()
encoded = 0
skipped = 0
errors = []

for i, sample in enumerate(ds):
    utt_id = sample['id']
    
    # Skip if already encoded
    if utt_id in existing:
        skipped += 1
        continue
    
    try:
        audio = sample['audio']
        wav = np.array(audio['array'], dtype=np.float32)
        sr = audio['sampling_rate']
        
        # Resample to SPARC's expected rate if needed
        if sr != sparc.sr:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=sparc.sr)
        
        # SPARC encode
        result = sparc.encode(wav)
        ema = result['ema'].squeeze()
        pitch = result['pitch'].squeeze()
        loudness = result['loudness'].squeeze()
        spk_emb = result['spk_emb'].squeeze()
        
        if isinstance(ema, torch.Tensor): ema = ema.cpu().numpy()
        if isinstance(pitch, torch.Tensor): pitch = pitch.cpu().numpy()
        if isinstance(loudness, torch.Tensor): loudness = loudness.cpu().numpy()
        if isinstance(spk_emb, torch.Tensor): spk_emb = spk_emb.cpu().numpy()
        
        # Save to Drive
        out_path = os.path.join(OUT_DIR, f'{utt_id}.npz')
        np.savez(
            out_path,
            ema=ema.astype(np.float32),
            pitch=pitch.astype(np.float32),
            loudness=loudness.astype(np.float32),
            spk_emb=spk_emb.astype(np.float32),
            style=str(sample['style']),
            text=str(sample['text']),
            speaker_id=str(sample['speaker_id']),
            utt_id=str(utt_id),
        )
        encoded += 1
        existing.add(utt_id)
        
    except Exception as e:
        errors.append((utt_id, str(e)))
        continue
    
    if (encoded + skipped) % 200 == 0:
        elapsed = time.time() - t0
        rate = encoded / max(1, elapsed)
        total_remaining = len(ds) - encoded - skipped
        eta = total_remaining / max(0.01, rate) / 3600
        print(f'[{encoded + skipped}/{len(ds)}] encoded={encoded} skipped={skipped} '
              f'errors={len(errors)} rate={rate:.1f}/s ETA={eta:.1f}h')

elapsed = time.time() - t0
print(f'\nDone! encoded={encoded} skipped={skipped} errors={len(errors)} time={elapsed/60:.0f}min')
if errors:
    print(f'First 5 errors: {errors[:5]}')

In [ ]:
# Cell 6: Verify output
import os, numpy as np
files = [f for f in os.listdir(OUT_DIR) if f.endswith('.npz')]
print(f'Total files: {len(files)}')

# Check a sample
d = np.load(os.path.join(OUT_DIR, files[0]), allow_pickle=True)
print(f'Keys: {list(d.keys())}')
print(f'ema: {d["ema"].shape}, pitch: {d["pitch"].shape}, spk_emb: {d["spk_emb"].shape}')
print(f'style: {d["style"]}, speaker: {d["speaker_id"]}')

# Style distribution
from collections import Counter
styles = Counter()
for f in files:
    d = np.load(os.path.join(OUT_DIR, f), allow_pickle=True)
    styles[str(d['style'])] += 1
print(f'\nStyle distribution:')
for s, c in styles.most_common():
    print(f'  {s}: {c}')